# 🔬 TikTok Acne — Metadata + Comments Enrichment

**Starting point:** `TikTok_Acne_Group2_Labeled.csv` (731 rows, Group 2 NLP fields filled)

**What this notebook fills:**
| Phase | Method | Fills |
|---|---|---|
| 1 | yt-dlp metadata scrape | `views`, `likes`, `shares`, `Posting date`, `creator_username`, `creator_followers` |
| 2 | yt-dlp comments fetch | `comment_text` (top 5 per video) |
| 3 | Compute | `engagement_rate` = likes/views×100 |
| 4 | Claude API | `comment_stance`, `comment_sentiment` |

**Still requires manual watching (not covered here):**
`creator_gender`, `creator_age_range`, `creator_race_visual`, `No._people_video`,
`video_shot_complexity`, `indoor_outdoor`, `video_format`, `Video_type`,
`face_visibility`, `condition_extent`, `condition_duration`, `condition_recurrence`

> ⚙️ Runtime → Change runtime type → **T4 GPU** (needed for nothing here but good practice)
>
> 🔑 Paste your Anthropic API key in **Cell 3**

In [ ]:
# ── STEP 1: Mount Google Drive ──────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/TikTok_Acne_Research'
DATA_DIR = f'{BASE_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)

print('✅ Drive mounted')
print(f'   Data → {DATA_DIR}')

In [ ]:
# ── STEP 2: Install dependencies ────────────────────────────────────
!pip install -q yt-dlp anthropic pandas tqdm numpy
print('✅ Dependencies installed')

In [ ]:
# ── STEP 3: Configuration ───────────────────────────────────────────

# 🔑 Paste your Anthropic API key here
ANTHROPIC_API_KEY = 'sk-ant-YOUR_KEY_HERE'

# File paths
CSV_INPUT    = f'{DATA_DIR}/TikTok_Acne_Group2_Labeled.csv'
CSV_ENRICHED = f'{DATA_DIR}/step1_metadata_enriched.csv'
CSV_FINAL    = f'{DATA_DIR}/TikTok_Acne_Final_Enriched.csv'

print('✅ Config set')
print(f'   API key: {"SET ✅" if ANTHROPIC_API_KEY != "sk-ant-YOUR_KEY_HERE" else "❌ MISSING"}')

In [ ]:
# ── STEP 4: Load CSV ────────────────────────────────────────────────
import pandas as pd, os

if not os.path.exists(CSV_INPUT):
    from google.colab import files
    print('CSV not found — upload TikTok_Acne_Group2_Labeled.csv now...')
    uploaded = files.upload()
    import shutil
    shutil.copy(list(uploaded.keys())[0], CSV_INPUT)

df = pd.read_csv(CSV_INPUT, dtype=str)  # read all as str to preserve IDs

# Fix creator_username — currently stored as numeric ID, extract @handle from URL
df['creator_username'] = df['video_url'].str.extract(r'@([^/]+)')

print(f'✅ Loaded: {len(df)} rows')
print(f'   creator_username sample: {df["creator_username"].head(3).tolist()}')
print(f'   video_url sample: {df["video_url"].head(1).tolist()}')
print()
print('EMPTY COLUMNS TO FILL:')
empty = [c for c in df.columns if (df[c].isna() | (df[c].astype(str).str.strip() == '')).all()]
print(f'  {empty}')

In [ ]:
# ── STEP 5: Scrape metadata via yt-dlp (views, likes, shares, date) ─
# Free — no API key needed, no video download
import subprocess, json, time, os, pandas as pd
from tqdm import tqdm

def scrape_meta(url, retries=2):
    """Fetch TikTok video metadata without downloading."""
    cmd = [
        'yt-dlp', '--no-download', '--dump-json', '--no-warnings',
        '--no-check-certificates',
        '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        '--add-header', 'Accept-Language:en-US,en;q=0.9',
        url
    ]
    for attempt in range(retries + 1):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
            if r.returncode == 0 and r.stdout.strip():
                d = json.loads(r.stdout.strip())
                return {
                    'video_url':          url,
                    'creator_followers':  d.get('channel_follower_count') or
                                          d.get('uploader_follower_count') or 0,
                    'views':              d.get('view_count') or 0,
                    'likes':              d.get('like_count') or 0,
                    'shares':             d.get('repost_count') or
                                          d.get('share_count') or 0,
                    'comments_scraped':   d.get('comment_count') or 0,
                    'Posting date':       d.get('upload_date') or '',
                    'scrape_status':      'success'
                }
        except Exception:
            if attempt < retries:
                time.sleep(2 ** attempt)
    return {'video_url': url, 'scrape_status': 'failed'}

# Resume from checkpoint
ckpt_path = CSV_ENRICHED + '.ckpt'
if os.path.exists(ckpt_path):
    ckpt_df   = pd.read_csv(ckpt_path, dtype=str)
    done_urls = set(ckpt_df['video_url'].tolist())
    results   = ckpt_df.to_dict('records')
    print(f'Resuming — {len(done_urls)} already scraped')
else:
    done_urls, results = set(), []

urls      = df['video_url'].tolist()
remaining = [u for u in urls if u not in done_urls]
print(f'Scraping metadata for {len(remaining)} URLs...')
print(f'Estimated time: ~40-60 mins\n')

for i, url in enumerate(tqdm(remaining)):
    results.append(scrape_meta(url))
    time.sleep(0.5)
    if (i + 1) % 50 == 0:
        pd.DataFrame(results).to_csv(ckpt_path, index=False)
        ok = sum(1 for r in results if r.get('scrape_status') == 'success')
        print(f'  [{len(results)}] {ok} succeeded')

# Save full checkpoint
pd.DataFrame(results).to_csv(ckpt_path, index=False)

meta_df = pd.DataFrame(results)
ok = (meta_df['scrape_status'] == 'success').sum()
print(f'\n✅ Metadata scrape complete: {ok}/{len(urls)} succeeded')
print(f'   Failed: {len(urls) - ok} (TikTok blocks/deleted videos)')

In [ ]:
# ── STEP 6: Fetch top comments via yt-dlp ───────────────────────────
# Fetches top 5 comments per video, concatenated as text
import subprocess, json, time, os, pandas as pd
from tqdm import tqdm

def fetch_comments(url, max_comments=5, retries=2):
    """Fetch top N comments from a TikTok video."""
    cmd = [
        'yt-dlp',
        '--no-download',
        '--dump-json',
        '--no-warnings',
        '--no-check-certificates',
        '--extractor-args', f'tiktok:max_comments={max_comments}',
        '--user-agent', 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        '--add-header', 'Accept-Language:en-US,en;q=0.9',
        url
    ]
    for attempt in range(retries + 1):
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=45)
            if r.returncode == 0 and r.stdout.strip():
                d = json.loads(r.stdout.strip())
                raw_comments = d.get('comments') or []
                if raw_comments:
                    # Take top N by like count, join as readable text
                    top = sorted(raw_comments,
                                 key=lambda c: c.get('like_count', 0),
                                 reverse=True)[:max_comments]
                    texts = [c.get('text', '').strip() for c in top if c.get('text', '').strip()]
                    return ' | '.join(texts) if texts else ''
        except Exception:
            if attempt < retries:
                time.sleep(2 ** attempt)
    return ''

# Resume from checkpoint
ckpt_comments = DATA_DIR + '/comments.ckpt'
if os.path.exists(ckpt_comments):
    ckpt_c    = pd.read_csv(ckpt_comments, dtype=str)
    done_c    = set(ckpt_c['video_url'].tolist())
    c_results = ckpt_c.to_dict('records')
    print(f'Resuming comments — {len(done_c)} already done')
else:
    done_c, c_results = set(), []

remaining_c = [u for u in df['video_url'].tolist() if u not in done_c]
print(f'Fetching comments for {len(remaining_c)} videos...')
print(f'Estimated time: ~60-90 mins\n')

for i, url in enumerate(tqdm(remaining_c)):
    comment_text = fetch_comments(url)
    c_results.append({'video_url': url, 'comment_text': comment_text})
    time.sleep(0.8)  # slightly longer delay — comment fetching is heavier
    if (i + 1) % 50 == 0:
        pd.DataFrame(c_results).to_csv(ckpt_comments, index=False)
        fetched = sum(1 for r in c_results if r.get('comment_text', ''))
        print(f'  [{len(c_results)}] {fetched} had comments')

pd.DataFrame(c_results).to_csv(ckpt_comments, index=False)
comments_df = pd.DataFrame(c_results)
fetched = (comments_df['comment_text'].fillna('') != '').sum()
print(f'\n✅ Comments fetched: {fetched}/{len(df)} had comments')

In [ ]:
# ── STEP 7: Merge scraped data + compute engagement_rate ─────────────
import pandas as pd, numpy as np, os

df = pd.read_csv(CSV_INPUT, dtype=str)

# Guard: fail clearly if Cell 5 or Cell 6 haven't run yet
meta_ckpt = CSV_ENRICHED + '.ckpt'
comm_ckpt = DATA_DIR + '/comments.ckpt'
if not os.path.exists(meta_ckpt):
    raise FileNotFoundError(f'Metadata checkpoint not found: {meta_ckpt}\nRun Cell 5 first.')
if not os.path.exists(comm_ckpt):
    raise FileNotFoundError(f'Comments checkpoint not found: {comm_ckpt}\nRun Cell 6 first.')

meta_df  = pd.read_csv(meta_ckpt, dtype=str)
comm_df  = pd.read_csv(comm_ckpt, dtype=str)

# Fix creator_username — always extract from URL (authoritative)
df['creator_username'] = df['video_url'].str.extract(r'@([^/]+)')

# Only keep successful scrapes for metadata
meta_ok = meta_df[meta_df['scrape_status'] == 'success'].copy()

# Columns to bring in from metadata scrape
# Drop columns that exist in df (we'll overwrite via map, not merge)
meta_cols = ['video_url', 'creator_followers', 'views', 'likes',
             'shares', 'comments_scraped', 'Posting date']
meta_ok = meta_ok[[c for c in meta_cols if c in meta_ok.columns]]

# Merge metadata — left join on video_url
# Drop existing empty columns from df before merge to avoid _x/_y suffixes
cols_to_overwrite = ['creator_followers', 'views', 'likes', 'shares', 'Posting date']
df = df.drop(columns=[c for c in cols_to_overwrite if c in df.columns], errors='ignore')
df = df.merge(meta_ok, on='video_url', how='left')

# Merge comments — left join on video_url
df = df.drop(columns=['comment_text'], errors='ignore')
df = df.merge(comm_df[['video_url', 'comment_text']], on='video_url', how='left')

# Compute engagement_rate = (likes / views) * 100, rounded to 2dp
# Convert to numeric first — coerce errors to NaN
df['views_num'] = pd.to_numeric(df['views'], errors='coerce')
df['likes_num'] = pd.to_numeric(df['likes'], errors='coerce')
df['engagement_rate'] = np.where(
    (df['views_num'] > 0) & df['views_num'].notna() & df['likes_num'].notna(),
    (df['likes_num'] / df['views_num'] * 100).round(2),
    np.nan
)
df = df.drop(columns=['views_num', 'likes_num'])

# Update comments column from scraped count where available
# (original 'comments' col had count data already — keep it, just rename scraped)
if 'comments_scraped' in df.columns:
    # Only overwrite where original is empty
    mask = df['comments'].isna() | (df['comments'] == '')
    df.loc[mask, 'comments'] = df.loc[mask, 'comments_scraped']
    df = df.drop(columns=['comments_scraped'])

# Save enriched CSV
df.to_csv(CSV_ENRICHED, index=False)

print('✅ Merge + compute complete')
print()
print('FILL RATES AFTER MERGE:')
for col in ['creator_username', 'creator_followers', 'views', 'likes',
            'shares', 'Posting date', 'engagement_rate', 'comment_text']:
    if col in df.columns:
        filled = (df[col].notna() & (df[col].astype(str) != '') & (df[col].astype(str) != 'nan')).sum()
        pct = filled / len(df) * 100
        status = '✅' if pct > 80 else '⚠️ ' if pct > 30 else '❌'
        print(f'  {status} {col:<25} {filled}/{len(df)} ({pct:.1f}%)')
print()
print(f'engagement_rate sample: {df["engagement_rate"].dropna().head(5).tolist()}')

In [ ]:
# ── STEP 8: Classify comment_stance + comment_sentiment via Claude ───
import anthropic, json, time, os, pandas as pd
from tqdm import tqdm

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """You are classifying TikTok comments for a dermatology misinformation study.
Given the top comments from a TikTok acne video, classify two things.
Respond ONLY with a valid JSON object — no markdown, no explanation.

comment_stance: Are comments supportive of the video's claims, or skeptical?
  Values: supportive | skeptical | neutral | mixed

comment_sentiment: What is the overall emotional tone of the comments?
  Values: positive | negative | neutral | mixed"""

DEFAULT = {'comment_stance': 'neutral', 'comment_sentiment': 'neutral'}

def classify_comments(comment_text, retries=2):
    """Classify stance and sentiment of comment text."""
    if not comment_text or str(comment_text).strip() in ('', 'nan'):
        return DEFAULT.copy()

    prompt = f"""COMMENTS (top 5 from a TikTok acne video, separated by |):
{str(comment_text)[:800]}

Return JSON with comment_stance and comment_sentiment."""

    for attempt in range(retries + 1):
        try:
            resp = client.messages.create(
                model='claude-haiku-4-5-20251001',  # cheaper model — simple classification
                max_tokens=60,
                system=SYSTEM_PROMPT,
                messages=[{'role': 'user', 'content': prompt}]
            )
            text = resp.content[0].text.strip()
            if text.startswith('```'):
                text = text.split('```')[1]
                if text.startswith('json'): text = text[4:]
            result = json.loads(text)
            # Validate values
            valid_stance     = {'supportive', 'skeptical', 'neutral', 'mixed'}
            valid_sentiment  = {'positive', 'negative', 'neutral', 'mixed'}
            return {
                'comment_stance':    result.get('comment_stance', 'neutral')
                                     if result.get('comment_stance') in valid_stance
                                     else 'neutral',
                'comment_sentiment': result.get('comment_sentiment', 'neutral')
                                     if result.get('comment_sentiment') in valid_sentiment
                                     else 'neutral'
            }
        except json.JSONDecodeError:
            if attempt < retries: time.sleep(1)
        except anthropic.RateLimitError:
            print('Rate limit — waiting 60s...')
            time.sleep(60)
        except Exception:
            if attempt < retries: time.sleep(2)
    return DEFAULT.copy()

df = pd.read_csv(CSV_ENRICHED, dtype=str)

# Only classify rows that have comment_text
has_comments = df['comment_text'].notna() & (df['comment_text'].str.strip() != '') & \
               (df['comment_text'].str.strip() != 'nan')
to_classify  = df[has_comments].copy()
print(f'Videos with comment_text: {len(to_classify)}')
print(f'Estimated API cost: ~${len(to_classify) * 0.0001:.2f} (using Haiku)\n')

# Resume from checkpoint
ckpt_cls = DATA_DIR + '/comment_classifications.ckpt'
if os.path.exists(ckpt_cls):
    done_cls  = pd.read_csv(ckpt_cls, dtype=str)
    done_urls = set(done_cls['video_url'].tolist())
    cls_results = done_cls.to_dict('records')
    print(f'Resuming — {len(done_urls)} already classified')
else:
    done_urls, cls_results = set(), []

remaining_cls = to_classify[~to_classify['video_url'].isin(done_urls)]

for _, row in tqdm(remaining_cls.iterrows(), total=len(remaining_cls)):
    comment_val = row['comment_text'] if pd.notna(row['comment_text']) else ''
    result = classify_comments(str(comment_val))
    result['video_url'] = row['video_url']
    cls_results.append(result)
    time.sleep(0.1)
    if len(cls_results) % 100 == 0:
        pd.DataFrame(cls_results).to_csv(ckpt_cls, index=False)
        print(f'  Checkpoint: {len(cls_results)} classified')

pd.DataFrame(cls_results).to_csv(ckpt_cls, index=False)

# Merge back into df
# Guard: if full resume, cls_results may be empty — read from checkpoint
if cls_results:
    cls_df = pd.DataFrame(cls_results)
else:
    cls_df = pd.read_csv(ckpt_cls, dtype=str)
df = df.drop(columns=['comment_stance', 'comment_sentiment'], errors='ignore')
df = df.merge(cls_df[['video_url', 'comment_stance', 'comment_sentiment']],
              on='video_url', how='left')

# Fill unclassified rows (no comments) with 'na'
df['comment_stance']    = df['comment_stance'].fillna('na')
df['comment_sentiment'] = df['comment_sentiment'].fillna('na')

df.to_csv(CSV_ENRICHED, index=False)

classified = (df['comment_stance'] != 'na').sum()
print(f'\n✅ Comment classification complete')
print(f'   Classified: {classified} | No comments: {len(df) - classified}')
print()
print('comment_stance distribution:')
print(df['comment_stance'].value_counts().to_string())
print()
print('comment_sentiment distribution:')
print(df['comment_sentiment'].value_counts().to_string())

In [ ]:
# ── STEP 9: Final assembly — reorder columns to match original schema ─
import pandas as pd

df = pd.read_csv(CSV_ENRICHED, dtype=str)

# Target column order — original 46 cols + new additions
col_order = [
    # Creator
    'creator_username', 'creator_gender', 'creator_age_range',
    'creator_race_visual', 'creator_type', 'creator_motivation',
    'creator_followers',
    # Video metadata
    'video_id', 'video_url', 'Posting date',
    # Visual features (manual)
    'No._people_video', 'video_shot_complexity', 'indoor_outdoor',
    'video_format', 'Video_type', 'face_visibility',
    # Content features
    'subject_of_experience', 'skin_condition_type',
    'condition_extent', 'condition_duration', 'condition_recurrence',
    'treatment_type', 'outcome_type', 'side_effects_mentioned',
    # Intent + promotion
    'video_intent', 'intent_clarity',
    'product_name', 'product_brand', 'sponsored_content',
    'affiliate_link_present', 'promotion_style',
    # Claims
    'before_after_claim', 'time_to_result_claim', 'unrealistic_claim_flag',
    # Engagement
    'views', 'likes', 'comments', 'shares', 'engagement_rate',
    # Comments
    'comment_text', 'comment_stance', 'comment_sentiment',
    # Classification
    'Missed_info', 'misinformation_label',
    'claim_type', 'claim_accuracy', 'evidence_present', 'label_confidence',
    # Raw text
    'caption', 'transcript'
]

# Only include columns that actually exist
final_cols = [c for c in col_order if c in df.columns]
# Add any extra columns not in the order list
extra_cols = [c for c in df.columns if c not in final_cols]
if extra_cols:
    print(f'Extra cols appended: {extra_cols}')
df_final = df[final_cols + extra_cols]
df_final.to_csv(CSV_FINAL, index=False)

print('✅ Final CSV assembled')
print(f'   Rows: {len(df_final)} | Columns: {len(df_final.columns)}')
print()
print('FINAL FILL RATES:')
for col in df_final.columns:
    filled = (df_final[col].notna() &
              (df_final[col].astype(str).str.strip() != '') &
              (df_final[col].astype(str).str.strip() != 'nan')).sum()
    pct = filled / len(df_final) * 100
    if pct == 0:
        status = '❌ (manual)'
    elif pct < 50:
        status = '⚠️ '
    else:
        status = '✅'
    print(f'  {status} {col:<35} {filled}/{len(df_final)} ({pct:.1f}%)')

In [ ]:
# ── STEP 10: Validate data quality ──────────────────────────────────
import pandas as pd

df = pd.read_csv(CSV_FINAL, dtype=str)

print('='*55)
print('DATA QUALITY CHECKS')
print('='*55)

# 1. Duplicate video_urls
dupes = df['video_url'].duplicated().sum()
print(f'Duplicate URLs: {dupes} {"✅" if dupes == 0 else "❌"}')

# 2. engagement_rate range check
er = pd.to_numeric(df['engagement_rate'], errors='coerce').dropna()
if len(er) > 0:
    print(f'engagement_rate range: {er.min():.2f}% – {er.max():.2f}% (mean: {er.mean():.2f}%)')
    outliers = (er > 100).sum()
    print(f'  engagement_rate > 100%: {outliers} rows {"⚠️" if outliers > 0 else "✅"}')
else:
    print('engagement_rate: no data scraped yet ⚠️')

# 3. creator_username — should be @handle not numeric
numeric_usernames = df['creator_username'].str.match(r'^\d+$', na=False).sum()
print(f'Numeric creator_usernames: {numeric_usernames} {"✅" if numeric_usernames == 0 else "❌"}')

# 4. Key classification columns — check for unexpected values
valid = {
    'misinformation_label': {'yes', 'no', 'partial'},
    'before_after_claim':   {'yes', 'no'},
    'label_confidence':     {'high', 'medium', 'low'},
    'claim_accuracy':       {'supported', 'partial', 'unsupported', 'false', 'na', 'anecdotal'},
    'comment_stance':       {'supportive', 'skeptical', 'neutral', 'mixed', 'na'},
    'comment_sentiment':    {'positive', 'negative', 'neutral', 'mixed', 'na'},
}
print()
print('VALUE VALIDATION:')
for col, allowed in valid.items():
    if col not in df.columns: continue
    unexpected = ~df[col].isin(allowed)
    n = unexpected.sum()
    bad_vals = df.loc[unexpected, col].unique()[:5] if n > 0 else []
    print(f'  {"✅" if n == 0 else "⚠️ "} {col:<25} {n} unexpected values {bad_vals if n > 0 else ""}')

# 5. Key stats
print()
print('KEY RESEARCH STATS:')
print(f'  Total videos:          {len(df)}')
print(f'  Misinformation (yes):  {(df["misinformation_label"]=="yes").sum()}')
print(f'  Misinformation (partial): {(df["misinformation_label"]=="partial").sum()}')
print(f'  Before/after videos:   {(df["before_after_claim"]=="yes").sum()}')
print(f'  Dermatologist videos:  {(df["creator_type"]=="dermatologist").sum()}')
print(f'  Influencer videos:     {(df["creator_type"]=="influencer").sum()}')
print(f'  Has engagement_rate:   {pd.to_numeric(df["engagement_rate"], errors="coerce").notna().sum()}')

In [ ]:
# ── STEP 11: Export ──────────────────────────────────────────────────
from google.colab import files
import pandas as pd

df = pd.read_csv(CSV_FINAL)

print(f'✅ Final dataset: {len(df)} rows × {len(df.columns)} columns')
print(f'   Saved to Drive: {CSV_FINAL}')
print('Downloading...')
files.download(CSV_FINAL)